
# 氮同位素独立模拟 notebook

基于 Kang et al. (2023) 和 Ma et al. (2025) 的双箱稳态氮循环模型。

**此文件完全自包含**，不依赖任何外部项目代码，仅需标准 Python 科学计算库：
- numpy
- scipy
- matplotlib
- pandas

## 模型功能
1. **正向模型**：从硝酸盐占比 $f_{assimilator}$ 计算沉积物 $\delta^{15}N_{sed}$
2. **反向模型**：从沉积物 $\delta^{15}N_{sed}$ 反演 $f_{assimilator}$
3. **蒙特卡洛不确定性分析**：评估分馏系数不确定性的影响
4. **关系曲线计算**：生成 $f$ vs $\delta^{15}N$ 曲线及置信区间


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from dataclasses import dataclass, field
from typing import Dict, Tuple, Optional

# 设置中文/英文绘图样式
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['xtick.major.width'] = 1.0
plt.rcParams['ytick.major.width'] = 1.0

print("✓ 库导入完成")


In [ ]:

# ============================================================
# 参数定义 (原 systems/n/parameters.py)
# ============================================================

@dataclass
class IsotopeParameters:
    """同位素参数数据类"""
    element: str = ""
    name: str = ""
    reference_standard: str = ""
    reference_ratios: Dict[str, float] = field(default_factory=dict)
    fractionation_factors: Dict[str, float] = field(default_factory=dict)
    end_members: Dict[str, Dict[str, float]] = field(default_factory=dict)
    reservoir_mass: float = 0.0
    input_fluxes: Dict[str, float] = field(default_factory=dict)
    output_fluxes: Dict[str, float] = field(default_factory=dict)


@dataclass
class NitrogenCycleFluxes:
    """氮循环通量 (Tg N/a)"""
    F_fix: float = 205.0
    F_total_burial: float = 25.0
    F_wcd: float = 140.0
    F_sd: float = 40.0
    
    @property
    def F_remin(self) -> float:
        return self.F_fix
    
    @property
    def F_total_denit(self) -> float:
        return self.F_wcd + self.F_sd


@dataclass
class FractionationFactors:
    """分馏系数 (‰)"""
    epsilon_fix: float = -0.5
    epsilon_wcd: float = -26.0
    epsilon_sd: float = 0.0
    
    def to_alpha(self, epsilon_name: str) -> float:
        epsilon = getattr(self, epsilon_name)
        return 1 + epsilon / 1000


# 标准物质参考值
NITROGEN_STANDARDS = {
    'air': {'name': 'Atmospheric N2', 'delta15': 0.0, 'description': '大气氮气，δ¹⁵N定义零点'}
}

# 分馏系数范围
FRACTIONATION_RANGES = {
    'fixation': {'min': -2.0, 'max': 1.0, 'description': '固氮作用分馏'},
    'water_column_denitrification': {'min': -30.0, 'max': -22.0, 'description': '水柱反硝化分馏'},
    'sedimentary_denitrification': {'min': 0.0, 'max': 0.0, 'description': '沉积反硝化分馏'},
}

# 氮循环通量参数
FLUX_PARAMETERS = {
    'modern': {
        'F_fix': 205.0, 'F_total_burial': 25.0, 'F_wcd': 140.0, 'F_sd': 40.0,
        'description': '现代海洋氮循环参数 (Kang et al. 2023)'
    },
    'early_triassic': {
        'F_fix': 205.0, 'F_total_burial': 25.0,
        'description': '早三叠世氮循环参数 (Ma et al. 2025)'
    },
    'neoproterozoic': {
        'F_fix': 205.0, 'F_total_burial': 25.0,
        'description': '新元古代氮循环参数 (Kang et al. 2023)'
    }
}

# 典型情景参数
SCENARIO_PARAMETERS = {
    'modern_oxic': {
        'f_assimilator_range': (0.3, 0.7),
        'delta15N_sed_range': (4.0, 6.0),
        'description': '现代氧化海洋'
    },
    'early_triassic_stage_I': {
        'f_assimilator_range': (0.0, 0.1),
        'delta15N_sed_range': (0.0, 2.0),
        'description': '早三叠世第I阶段 (缺氧高温)'
    },
    'early_triassic_stage_II': {
        'f_assimilator_range': (0.15, 0.25),
        'delta15N_sed_range': (3.0, 5.0),
        'description': '早三叠世第II阶段 (氧化降温)'
    },
    'early_triassic_stage_III': {
        'f_assimilator_range': (0.05, 0.15),
        'delta15N_sed_range': (1.0, 3.0),
        'description': '早三叠世第III阶段 (再缺氧)'
    },
    'neoproterozoic_pre_800Ma': {
        'f_assimilator_range': (0.05, 0.20),
        'delta15N_sed_range': (0.5, 3.0),
        'description': '新元古代早期 (<800Ma)'
    },
    'neoproterozoic_post_800Ma': {
        'f_assimilator_range': (0.15, 0.35),
        'delta15N_sed_range': (3.0, 6.0),
        'description': '新元古代晚期 (>800Ma)'
    },
    'anoxic_nitrate_depleted': {
        'f_assimilator_range': (0.0, 0.1),
        'delta15N_sed_range': (-1.0, 2.0),
        'description': '缺氧硝酸盐匮乏环境'
    }
}


def get_n_parameters(scenario: str = 'modern') -> IsotopeParameters:
    """获取氮同位素参数"""
    base_params = {
        'element': 'n', 'name': 'Nitrogen',
        'reference_standard': 'Air-N2',
        'reference_ratios': {'15/14': 0.003676},
    }
    flux_params = FLUX_PARAMETERS.get(scenario, FLUX_PARAMETERS['modern'])
    return IsotopeParameters(
        **base_params,
        reservoir_mass=5.7e16,
        input_fluxes={
            'fixation': flux_params['F_fix'],
            'atmospheric_deposition': 5.0,
            'river_input': 20.0,
        },
        output_fluxes={
            'water_column_denitrification': flux_params.get('F_wcd', 140.0),
            'sedimentary_denitrification': flux_params.get('F_sd', 40.0),
            'burial': flux_params['F_total_burial'],
        },
        end_members={
            'atmosphere': {'delta15': 0.0, 'description': '大气N₂ (定义标准)'},
            'nitrogen_fixer': {'delta15': -1.0, 'range': (-2.0, 1.0), 'description': '固氮生物 (蓝细菌等)'},
            'nitrate_assimilator': {'delta15': 5.0, 'range': (3.0, 8.0), 'description': '硝酸盐同化生物 (真核藻类等)'},
            'ammonium_dominant_ocean': {'delta15': -0.5, 'range': (-1.0, 1.0), 'description': '铵主导海洋 (缺氧环境)'},
            'nitrate_dominant_ocean': {'delta15': 5.0, 'range': (3.0, 7.0), 'description': '硝酸盐主导海洋 (氧化环境)'}
        },
        fractionation_factors={
            'epsilon_fixation': -0.5,
            'epsilon_fixation_min': -2.0,
            'epsilon_fixation_max': 1.0,
            'epsilon_wcd': -26.0,
            'epsilon_wcd_min': -30.0,
            'epsilon_wcd_max': -22.0,
            'epsilon_sd': 0.0,
        }
    )


def get_scenario_info(scenario_name: str) -> Dict:
    """获取情景信息"""
    return SCENARIO_PARAMETERS.get(scenario_name, {
        'description': '未知情景',
        'f_assimilator_range': (0.0, 1.0),
        'delta15N_sed_range': (-5.0, 10.0)
    })


def get_fractionation_ranges() -> Dict[str, Tuple[float, float]]:
    """获取分馏系数范围"""
    return {
        'fixation': (FRACTIONATION_RANGES['fixation']['min'], FRACTIONATION_RANGES['fixation']['max']),
        'water_column_denitrification': (FRACTIONATION_RANGES['water_column_denitrification']['min'],
                                         FRACTIONATION_RANGES['water_column_denitrification']['max']),
        'sedimentary_denitrification': (FRACTIONATION_RANGES['sedimentary_denitrification']['min'],
                                        FRACTIONATION_RANGES['sedimentary_denitrification']['max'])
    }


print("✓ 参数定义完成")


In [ ]:

# ============================================================
# 核心模型类 (原 systems/n/model.py)
# ============================================================

class NIsotopeSystem:
    """
    氮同位素双箱稳态模型
    
    基于 Kang et al. (2023) 和 Ma et al. (2025)
    """
    
    ELEMENT = 'n'
    NAME = 'Nitrogen'
    ISOTOPES = ['14N', '15N']
    DELTA15N_ATMOSPHERE = 0.0
    
    def __init__(self, parameters: Optional[IsotopeParameters] = None,
                 scenario: str = 'modern'):
        self.scenario = scenario
        self.params = parameters or get_n_parameters(scenario)
        self.fluxes = self._init_fluxes()
        self.fractionation = self._init_fractionation()
    
    def _init_fluxes(self) -> NitrogenCycleFluxes:
        params = self.params
        return NitrogenCycleFluxes(
            F_fix=params.input_fluxes.get('fixation', 205.0),
            F_total_burial=params.output_fluxes.get('burial', 25.0),
            F_wcd=params.output_fluxes.get('water_column_denitrification', 140.0),
            F_sd=params.output_fluxes.get('sedimentary_denitrification', 40.0)
        )
    
    def _init_fractionation(self) -> FractionationFactors:
        ff = self.params.fractionation_factors
        return FractionationFactors(
            epsilon_fix=ff.get('epsilon_fixation', -0.5),
            epsilon_wcd=ff.get('epsilon_wcd', -26.0),
            epsilon_sd=ff.get('epsilon_sd', 0.0)
        )
    
    def calculate_reservoir_isotopes(self,
                                     f_assimilator: float,
                                     epsilon_fix: Optional[float] = None,
                                     epsilon_wcd: Optional[float] = None,
                                     epsilon_sd: Optional[float] = None) -> Dict[str, float]:
        """计算储库同位素组成 (方程 3, 4)"""
        eps_fix = epsilon_fix if epsilon_fix is not None else self.fractionation.epsilon_fix
        eps_wcd = epsilon_wcd if epsilon_wcd is not None else self.fractionation.epsilon_wcd
        eps_sd = epsilon_sd if epsilon_sd is not None else self.fractionation.epsilon_sd
        
        # 铵储库 (方程 3)
        delta15N_ammonium = self.DELTA15N_ATMOSPHERE + eps_fix
        
        # 水柱反硝化随 f 增加而减少
        F_wcd_max = self.fluxes.F_wcd * 1.5
        F_wcd_effective = F_wcd_max * (1 - f_assimilator)
        F_sd_effective = self.fluxes.F_sd
        F_remin = self.fluxes.F_fix
        
        # 硝酸盐储库 (方程 4)
        weighted_epsilon = (F_wcd_effective * eps_wcd + F_sd_effective * eps_sd) / F_remin
        delta15N_nitrate = delta15N_ammonium - weighted_epsilon
        
        return {
            'delta15N_ammonium': delta15N_ammonium,
            'delta15N_nitrate': delta15N_nitrate
        }
    
    def forward_model(self,
                     f_assimilator: float,
                     epsilon_fix: Optional[float] = None,
                     epsilon_wcd: Optional[float] = None,
                     epsilon_sd: Optional[float] = None) -> float:
        """正向模型：f_assimilator -> δ¹⁵N_sed (方程 6)"""
        reservoirs = self.calculate_reservoir_isotopes(
            f_assimilator, epsilon_fix, epsilon_wcd, epsilon_sd
        )
        delta15N_ammonium = reservoirs['delta15N_ammonium']
        delta15N_nitrate = reservoirs['delta15N_nitrate']
        
        delta15N_sed = ((1 - f_assimilator) * delta15N_ammonium +
                       f_assimilator * delta15N_nitrate)
        return delta15N_sed
    
    def inverse_model(self,
                     delta15N_sed: float,
                     epsilon_fix: Optional[float] = None,
                     epsilon_wcd: Optional[float] = None,
                     epsilon_sd: Optional[float] = None,
                     f_range: Tuple[float, float] = (0.0, 0.48)) -> Dict:
        """反向模型：δ¹⁵N_sed -> f_assimilator"""
        def objective(f):
            calculated = self.forward_model(f, epsilon_fix, epsilon_wcd, epsilon_sd)
            return (calculated - delta15N_sed) ** 2
        
        result = minimize_scalar(objective, bounds=f_range, method='bounded')
        f_optimal = result.x
        delta_calc = self.forward_model(f_optimal, epsilon_fix, epsilon_wcd, epsilon_sd)
        
        return {
            'f_assimilator': float(f_optimal),
            'delta15N_sed_calculated': float(delta_calc),
            'residual': float(delta_calc - delta15N_sed)
        }
    
    def monte_carlo_simulation(self,
                              f_assimilator: float,
                              n_samples: int = 10000,
                              epsilon_fix_range: Tuple[float, float] = (-2.0, 1.0),
                              epsilon_wcd_range: Tuple[float, float] = (-30.0, -22.0)) -> Dict:
        """蒙特卡洛模拟评估不确定性"""
        epsilon_fix_samples = np.random.uniform(
            epsilon_fix_range[0], epsilon_fix_range[1], n_samples
        )
        epsilon_wcd_samples = np.random.uniform(
            epsilon_wcd_range[0], epsilon_wcd_range[1], n_samples
        )
        
        delta15N_sed_samples = np.array([
            self.forward_model(f_assimilator, eps_fix, eps_wcd)
            for eps_fix, eps_wcd in zip(epsilon_fix_samples, epsilon_wcd_samples)
        ])
        
        mean_val = np.mean(delta15N_sed_samples)
        std_val = np.std(delta15N_sed_samples)
        median_val = np.median(delta15N_sed_samples)
        ci68 = (np.percentile(delta15N_sed_samples, 16),
                np.percentile(delta15N_sed_samples, 84))
        ci95 = (np.percentile(delta15N_sed_samples, 2.5),
                np.percentile(delta15N_sed_samples, 97.5))
        
        return {
            'delta15N_sed_mean': float(mean_val),
            'delta15N_sed_std': float(std_val),
            'delta15N_sed_median': float(median_val),
            'delta15N_sed_ci68': (float(ci68[0]), float(ci68[1])),
            'delta15N_sed_ci95': (float(ci95[0]), float(ci95[1])),
            'samples': delta15N_sed_samples
        }
    
    def calculate_f_assimilator_curve(self,
                                     f_range: Tuple[float, float] = (0.0, 1.0),
                                     n_points: int = 100,
                                     n_monte_carlo: int = 1000) -> Dict:
        """计算 f_assimilator 与 δ¹⁵N_sed 的关系曲线"""
        f_values = np.linspace(f_range[0], f_range[1], n_points)
        
        mean_curve = []
        ci68_lower = []
        ci68_upper = []
        ci95_lower = []
        ci95_upper = []
        
        for f in f_values:
            mc_result = self.monte_carlo_simulation(f, n_samples=n_monte_carlo)
            mean_curve.append(mc_result['delta15N_sed_mean'])
            ci68_lower.append(mc_result['delta15N_sed_ci68'][0])
            ci68_upper.append(mc_result['delta15N_sed_ci68'][1])
            ci95_lower.append(mc_result['delta15N_sed_ci95'][0])
            ci95_upper.append(mc_result['delta15N_sed_ci95'][1])
        
        return {
            'f_assimilator': f_values,
            'delta15N_sed_mean': np.array(mean_curve),
            'delta15N_sed_ci68_lower': np.array(ci68_lower),
            'delta15N_sed_ci68_upper': np.array(ci68_upper),
            'delta15N_sed_ci95_lower': np.array(ci95_lower),
            'delta15N_sed_ci95_upper': np.array(ci95_upper)
        }
    
    def get_model_info(self) -> Dict:
        return {
            'element': self.ELEMENT,
            'name': self.NAME,
            'fluxes': {
                'F_fix': self.fluxes.F_fix,
                'F_total_burial': self.fluxes.F_total_burial,
                'F_wcd': self.fluxes.F_wcd,
                'F_sd': self.fluxes.F_sd
            },
            'fractionation': {
                'epsilon_fix': self.fractionation.epsilon_fix,
                'epsilon_wcd': self.fractionation.epsilon_wcd,
                'epsilon_sd': self.fractionation.epsilon_sd
            }
        }


print("✓ NIsotopeSystem 类定义完成")



## 1. 正向模型示例

给定 $f_{assimilator}$（硝酸盐同化埋藏比例），计算沉积物 $\delta^{15}N_{sed}$。

In [ ]:

# 创建模型实例
n_modern = NIsotopeSystem(scenario='modern')
n_neoproterozoic = NIsotopeSystem(scenario='neoproterozoic')
n_triassic = NIsotopeSystem(scenario='early_triassic')

print("=== 正向模型计算示例 ===\n")
print(f"{'情景':<20} {'f':<8} {'δ¹⁵N_sed (‰)':<12}")
print("-" * 45)

for scenario, model in [("现代海洋", n_modern), ("新元古代", n_neoproterozoic), ("早三叠世", n_triassic)]:
    for f in [0.0, 0.2, 0.48, 0.7, 1.0]:
        delta = model.forward_model(f)
        label = scenario if f == 0.0 else ""
        print(f"{label:<20} {f:<8.2f} {delta:<+12.2f}")
    print()



## 2. 反向模型示例

从观测到的沉积物 $\delta^{15}N$ 反演 $f_{assimilator}$。

In [ ]:

# 新元古代观测数据反演
observations_pre = [0.5, 1.0, 1.5, 2.0]   # 800 Ma 之前
observations_post = [3.0, 4.0, 5.0, 6.0]  # 800 Ma 之后

print("=== 反向反演: δ¹⁵N → f_assimilator (新元古代) ===\n")

print("[800 Ma 之前 - 缺氧硝酸盐匮乏]")
print(f"{'观测 δ¹⁵N':<12} {'f_assimilator':<15} {'解释'}")
print("-" * 60)
results_pre = []
for delta_obs in observations_pre:
    result = n_neoproterozoic.inverse_model(delta15N_sed=delta_obs, f_range=(0.0, 0.30))
    f_inv = result['f_assimilator']
    results_pre.append((delta_obs, f_inv))
    interp = "极度匮乏" if f_inv < 0.1 else ("严重受限" if f_inv < 0.2 else "轻度受限")
    print(f"{delta_obs:<+12.2f} {f_inv:<15.3f} {interp}")

print("\n[800 Ma 之后 - 氧化硝酸盐充足]")
print(f"{'观测 δ¹⁵N':<12} {'f_assimilator':<15} {'解释'}")
print("-" * 60)
results_post = []
for delta_obs in observations_post:
    result = n_neoproterozoic.inverse_model(delta15N_sed=delta_obs, f_range=(0.0, 0.50))
    f_inv = result['f_assimilator']
    results_post.append((delta_obs, f_inv))
    interp = "轻度受限" if f_inv < 0.25 else "硝酸盐充足"
    print(f"{delta_obs:<+12.2f} {f_inv:<15.3f} {interp}")



## 3. 关系曲线计算（带蒙特卡洛不确定性）

计算 $f_{assimilator}$ 与 $\delta^{15}N_{sed}$ 的完整关系曲线，包含 68% 和 95% 置信区间。

In [ ]:

# 新元古代关系曲线
curve_neo = n_neoproterozoic.calculate_f_assimilator_curve(
    f_range=(0.0, 1.0), n_points=100, n_monte_carlo=5000
)

# 找到峰值点
max_idx = np.argmax(curve_neo['delta15N_sed_mean'])
f_peak = curve_neo['f_assimilator'][max_idx]
delta_peak = curve_neo['delta15N_sed_mean'][max_idx]

print(f"峰值 δ¹⁵N: {delta_peak:+.2f}‰")
print(f"峰值位置 f: {f_peak:.3f}")

# 早三叠世关系曲线
curve_triassic = n_triassic.calculate_f_assimilator_curve(
    f_range=(0.0, 0.5), n_points=50, n_monte_carlo=5000
)



## 4. 绘图：f_assimilator vs δ¹⁵N 关系曲线

In [ ]:

fig, ax = plt.subplots(figsize=(7, 5))

# 均值曲线
ax.plot(curve_neo['f_assimilator'], curve_neo['delta15N_sed_mean'],
        'b-', linewidth=1.5, label='Mean δ¹⁵N$_{sed}$')

# 置信区间
ax.fill_between(curve_neo['f_assimilator'],
                curve_neo['delta15N_sed_ci95_lower'],
                curve_neo['delta15N_sed_ci95_upper'],
                alpha=0.2, color='blue', label='95% CI')
ax.fill_between(curve_neo['f_assimilator'],
                curve_neo['delta15N_sed_ci68_lower'],
                curve_neo['delta15N_sed_ci68_upper'],
                alpha=0.3, color='blue', label='68% CI')

# 峰值点
ax.plot(f_peak, delta_peak, 'r*', markersize=14,
        label=f'Peak (f={f_peak:.2f}, δ¹⁵N={delta_peak:.1f}‰)', zorder=5)

# 现代海洋参考点
ax.axvspan(0.3, 0.7, alpha=0.1, color='green', label='Modern ocean range')
ax.axvspan(0.0, 0.1, alpha=0.1, color='red', label='Anoxic range')

# 反向反演观测点
pre_f_inv = [f for _, f in results_pre]
pre_delta = [d for d, _ in results_pre]
post_f_inv = [f for _, f in results_post]
post_delta = [d for d, _ in results_post]

ax.scatter(pre_f_inv, pre_delta, c='red', s=80, marker='o',
           edgecolors='black', linewidth=0.5, label='pre-800 Ma obs.', zorder=5)
ax.scatter(post_f_inv, post_delta, c='green', s=80, marker='s',
           edgecolors='black', linewidth=0.5, label='post-800 Ma obs.', zorder=5)

ax.set_xlabel('f$_{assimilator}$ (nitrate assimilation fraction)', fontsize=12)
ax.set_ylabel('δ¹⁵N$_{sed}$ (‰)', fontsize=12)
ax.set_title('Nitrogen Isotope Model: f$_{assimilator}$ vs δ¹⁵N$_{sed}$', fontsize=12)
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(-2, 8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('nitrogen_curve_neoproterozoic.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ 图片已保存: nitrogen_curve_neoproterozoic.png")



## 5. 早三叠世阶段对比

In [ ]:

stages = {
    'Stage I': get_scenario_info('early_triassic_stage_I'),
    'Stage II': get_scenario_info('early_triassic_stage_II'),
    'Stage III': get_scenario_info('early_triassic_stage_III'),
}

stage_data = []
stage_colors = {'Stage I': '#d62728', 'Stage II': '#2ca02c', 'Stage III': '#ff7f0e'}

for stage_name, stage_info in stages.items():
    f_min, f_max = stage_info['f_assimilator_range']
    delta_min = n_triassic.forward_model(f_min)
    delta_max = n_triassic.forward_model(f_max)
    f_mean = (f_min + f_max) / 2
    delta_mean = (delta_min + delta_max) / 2
    stage_data.append({
        'name': stage_name,
        'f_range': (f_min, f_max),
        'delta_range': (delta_min, delta_max),
        'f_mean': f_mean,
        'delta_mean': delta_mean,
        'color': stage_colors[stage_name]
    })

fig, ax = plt.subplots(figsize=(7, 5))

# 绘制模型曲线
ax.plot(curve_triassic['f_assimilator'], curve_triassic['delta15N_sed_mean'],
        'k-', linewidth=1.5, alpha=0.5, label='Model curve')

# 标记各阶段
for sd in stage_data:
    ax.axvspan(sd['f_range'][0], sd['f_range'][1],
               alpha=0.15, color=sd['color'])
    ax.scatter(sd['f_mean'], sd['delta_mean'], c=sd['color'], s=150,
               marker='D', edgecolors='black', linewidth=1,
               label=f"{sd['name']}: f={sd['f_mean']:.2f}, δ¹⁵N={sd['delta_mean']:.1f}‰",
               zorder=5)

ax.set_xlabel('f$_{assimilator}$', fontsize=12)
ax.set_ylabel('δ¹⁵N$_{sed}$ (‰)', fontsize=12)
ax.set_title('Early Triassic Nitrogen Isotope Stages', fontsize=12)
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(0, 0.5)
ax.set_ylim(-1, 6)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('nitrogen_triassic_stages.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ 图片已保存: nitrogen_triassic_stages.png")



## 6. 模拟时间序列演化

In [ ]:

ages = np.array([251.9, 251.5, 251.0, 250.5, 250.0, 249.5, 249.0, 248.5, 248.0, 247.5])

np.random.seed(42)
base_delta = np.piecewise(ages,
    [ages > 250.5, (ages <= 250.5) & (ages > 248.8), ages <= 248.8],
    [lambda x: 1.0, lambda x: 4.0, lambda x: 2.0])
noise = np.random.normal(0, 0.3, len(ages))
observed_delta = base_delta + noise

inferred_f = []
for delta in observed_delta:
    result = n_triassic.inverse_model(delta15N_sed=delta, f_range=(0.0, 0.50))
    inferred_f.append(result['f_assimilator'])
inferred_f = np.array(inferred_f)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# δ¹⁵N 时间序列
ax1.plot(ages, observed_delta, 'o-', color='steelblue', markersize=6,
         linewidth=1.5, label='Observed δ¹⁵N')
ax1.axvspan(250.5, 251.9, alpha=0.15, color='#d62728', label='Stage I')
ax1.axvspan(248.8, 250.5, alpha=0.15, color='#2ca02c', label='Stage II')
ax1.axvspan(247.5, 248.8, alpha=0.15, color='#ff7f0e', label='Stage III')
ax1.set_ylabel('δ¹⁵N$_{sed}$ (‰)', fontsize=11)
ax1.set_title('Simulated Early Triassic Nitrogen Isotope Evolution', fontsize=12)
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, alpha=0.3)

# f_assimilator 时间序列
ax2.plot(ages, inferred_f, 's-', color='darkgreen', markersize=6,
         linewidth=1.5, label='Inferred f$_{assimilator}$')
ax2.axvspan(250.5, 251.9, alpha=0.15, color='#d62728')
ax2.axvspan(248.8, 250.5, alpha=0.15, color='#2ca02c')
ax2.axvspan(247.5, 248.8, alpha=0.15, color='#ff7f0e')
ax2.set_xlabel('Age (Ma)', fontsize=11)
ax2.set_ylabel('f$_{assimilator}$', fontsize=11)
ax2.set_title('Inferred Nitrate Availability', fontsize=12)
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('nitrogen_triassic_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ 图片已保存: nitrogen_triassic_timeseries.png")



## 7. 蒙特卡洛不确定性分布

In [ ]:

stage_f_values = {'Stage I': 0.05, 'Stage II': 0.20, 'Stage III': 0.10}
mc_results_list = []

for stage_name, f in stage_f_values.items():
    mc_result = n_triassic.monte_carlo_simulation(
        f_assimilator=f, n_samples=10000,
        epsilon_fix_range=(-2.0, 1.0), epsilon_wcd_range=(-30.0, -22.0)
    )
    mc_results_list.append({
        'stage': stage_name, 'f': f,
        'mean': mc_result['delta15N_sed_mean'],
        'std': mc_result['delta15N_sed_std'],
        'ci68': mc_result['delta15N_sed_ci68'],
        'ci95': mc_result['delta15N_sed_ci95'],
        'samples': mc_result['samples']
    })

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#d62728', '#2ca02c', '#ff7f0e']

for idx, (result, color) in enumerate(zip(mc_results_list, colors)):
    ax = axes[idx]
    samples = result['samples']
    ax.hist(samples, bins=50, density=True, alpha=0.6, color=color,
            edgecolor='black', linewidth=0.5)
    ax.axvline(result['mean'], color='black', linestyle='--', linewidth=2,
               label=f"Mean: {result['mean']:.2f}‰")
    ax.axvspan(result['ci68'][0], result['ci68'][1], alpha=0.2, color='blue',
               label=f"68% CI: [{result['ci68'][0]:.2f}, {result['ci68'][1]:.2f}]")
    ax.set_title(f"{result['stage']}\n(f={result['f']:.2f})", fontsize=11)
    ax.set_xlabel('δ¹⁵N$_{sed}$ (‰)', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('nitrogen_monte_carlo.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ 图片已保存: nitrogen_monte_carlo.png")



## 8. 综合对比：Kang et al. (2023) vs Ma et al. (2025)

In [ ]:

comparison_data = [
    ('Kang 2023\npre-800Ma', n_neoproterozoic, 0.11, 'Neoproterozoic\nearly (anoxic)'),
    ('Kang 2023\npost-800Ma', n_neoproterozoic, 0.35, 'Neoproterozoic\nlate (oxic)'),
    ('Ma 2025\nStage I', n_triassic, 0.05, 'Early Triassic I\n(extreme anoxia)'),
    ('Ma 2025\nStage II', n_triassic, 0.20, 'Early Triassic II\n(transient oxic)'),
    ('Ma 2025\nStage III', n_triassic, 0.10, 'Early Triassic III\n(re-anoxia)'),
]

labels = [d[0] for d in comparison_data]
deltas = [d[1].forward_model(d[2]) for d in comparison_data]
f_values = [d[2] for d in comparison_data]
conditions = [d[3] for d in comparison_data]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# f_assimilator 对比
bar_colors = ['#d62728', '#2ca02c', '#d62728', '#2ca02c', '#ff7f0e']
bars1 = ax1.bar(range(len(labels)), f_values, color=bar_colors, edgecolor='black', linewidth=0.5)
ax1.set_xticks(range(len(labels)))
ax1.set_xticklabels(labels, fontsize=9)
ax1.set_ylabel('f$_{assimilator}$', fontsize=11)
ax1.set_title('Nitrate Assimilation Fraction', fontsize=12)
ax1.set_ylim(0, 0.5)
ax1.grid(True, alpha=0.3, axis='y')
for i, (bar, val) in enumerate(zip(bars1, f_values)):
    ax1.text(bar.get_x() + bar.get_width()/2., val + 0.01,
             f'{val:.2f}', ha='center', va='bottom', fontsize=9)

# δ¹⁵N 对比
bars2 = ax2.bar(range(len(labels)), deltas, color=bar_colors, edgecolor='black', linewidth=0.5)
ax2.set_xticks(range(len(labels)))
ax2.set_xticklabels(labels, fontsize=9)
ax2.set_ylabel('δ¹⁵N$_{sed}$ (‰)', fontsize=11)
ax2.set_title('Sedimentary δ¹⁵N', fontsize=12)
ax2.set_ylim(0, 7)
ax2.grid(True, alpha=0.3, axis='y')
for i, (bar, val) in enumerate(zip(bars2, deltas)):
    ax2.text(bar.get_x() + bar.get_width()/2., val + 0.1,
             f'{val:.1f}‰', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('nitrogen_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ 图片已保存: nitrogen_comparison.png")



## 9. 数据导出为 CSV/Excel

In [ ]:

# 导出关系曲线数据
df_curve = pd.DataFrame({
    'f_assimilator': curve_neo['f_assimilator'],
    'delta15N_mean': curve_neo['delta15N_sed_mean'],
    'delta15N_ci68_lower': curve_neo['delta15N_sed_ci68_lower'],
    'delta15N_ci68_upper': curve_neo['delta15N_sed_ci68_upper'],
    'delta15N_ci95_lower': curve_neo['delta15N_sed_ci95_lower'],
    'delta15N_ci95_upper': curve_neo['delta15N_sed_ci95_upper']
})
df_curve.to_csv('nitrogen_curve_data.csv', index=False)
print("✓ 曲线数据已保存: nitrogen_curve_data.csv")

# 导出蒙特卡洛结果
mc_df = pd.DataFrame({
    'stage': [r['stage'] for r in mc_results_list],
    'f_assimilator': [r['f'] for r in mc_results_list],
    'delta15N_mean': [r['mean'] for r in mc_results_list],
    'delta15N_std': [r['std'] for r in mc_results_list],
    'ci68_lower': [r['ci68'][0] for r in mc_results_list],
    'ci68_upper': [r['ci68'][1] for r in mc_results_list],
    'ci95_lower': [r['ci95'][0] for r in mc_results_list],
    'ci95_upper': [r['ci95'][1] for r in mc_results_list],
})
mc_df.to_csv('nitrogen_monte_carlo_summary.csv', index=False)
print("✓ 蒙特卡洛结果已保存: nitrogen_monte_carlo_summary.csv")

# 导出对比数据
comp_df = pd.DataFrame({
    'label': labels,
    'f_assimilator': f_values,
    'delta15N_sed': deltas,
    'condition': conditions
})
comp_df.to_csv('nitrogen_comparison_data.csv', index=False)
print("✓ 对比数据已保存: nitrogen_comparison_data.csv")



---

## 使用说明

### 修改参数
可以直接修改模型实例的分馏系数：
```python
n_custom = NIsotopeSystem(scenario='modern')
n_custom.fractionation.epsilon_wcd = -28.0  # 修改水柱反硝化分馏
n_custom.fractionation.epsilon_fix = -1.0   # 修改固氮分馏
delta = n_custom.forward_model(f_assimilator=0.3)
```

### 添加自己的数据
```python
my_delta_values = [2.5, 3.0, 4.5]
for d in my_delta_values:
    result = n_neoproterozoic.inverse_model(delta15N_sed=d, f_range=(0.0, 0.5))
    print(f"δ¹⁵N={d} -> f={result['f_assimilator']:.3f}")
```

### 运行需要的环境
```bash
pip install numpy scipy matplotlib pandas
```


---

## 10. 实测数据模拟分析

使用项目自带的 `data/nitrogen_with_uncertainty.xlsx` 进行完整的反向模拟与不确定性分析。

分析流程：
1. 读取实测 δ¹⁵N 数据及其标准差
2. 对每个样品做反向反演，得到中心 f 值
3. 通过蒙特卡洛采样传播 δ¹⁵N 观测不确定性到 f_assimilator
4. 绘制年龄-硝酸盐可用性演化图
5. 在理论关系曲线上标注实测样品


In [ ]:
# 读取实测数据
df_data = pd.read_excel('data/nitrogen_with_uncertainty.xlsx')
print(df_data.to_string(index=False))
print(f"\n共 {len(df_data)} 个样品")
print(f"年龄范围: {df_data['age_ma'].min():.0f} – {df_data['age_ma'].max():.0f} Ma")
print(f"δ¹⁵N 范围: {df_data['delta15N'].min():.1f} – {df_data['delta15N'].max():.1f} ‰")


In [ ]:
# ============================================================
# 对每个样品进行反向反演 + 观测不确定性蒙特卡洛传播
# ============================================================

n_mc_obs = 2000  # 观测不确定性蒙特卡洛采样数

results = []
for _, row in df_data.iterrows():
    delta = row['delta15N']
    std = row['delta15N_std']
    age = row['age_ma']
    sid = row['sample_id']
    
    # 根据年龄自动选择合理的 f 范围
    # >800 Ma: 新元古代早期，缺氧，f 范围较窄
    # <=800 Ma: 新元古代晚期，氧化，f 范围较宽
    if age > 800:
        f_range = (0.0, 0.35)
    else:
        f_range = (0.0, 0.60)
    
    # 中心值反演
    central = n_neoproterozoic.inverse_model(delta, f_range=f_range)
    
    # 蒙特卡洛：从观测不确定性 N(delta, std²) 采样
    delta_samples = np.random.normal(delta, std, n_mc_obs)
    f_samples = []
    for d_samp in delta_samples:
        # 限制在物理合理范围内，避免极端值导致优化失败
        if d_samp < -2 or d_samp > 10:
            continue
        try:
            res = n_neoproterozoic.inverse_model(d_samp, f_range=f_range)
            f_samples.append(res['f_assimilator'])
        except Exception:
            continue
    
    f_samples = np.array(f_samples)
    f_mean = float(np.mean(f_samples))
    f_std = float(np.std(f_samples))
    f_ci68 = (float(np.percentile(f_samples, 16)),
              float(np.percentile(f_samples, 84)))
    f_ci95 = (float(np.percentile(f_samples, 2.5)),
              float(np.percentile(f_samples, 97.5)))
    
    results.append({
        'sample_id': sid,
        'age_ma': age,
        'delta15N': delta,
        'delta15N_std': std,
        'lithology': row['lithology'],
        'f_central': float(central['f_assimilator']),
        'f_mean': f_mean,
        'f_std': f_std,
        'f_ci68_lower': f_ci68[0],
        'f_ci68_upper': f_ci68[1],
        'f_ci95_lower': f_ci95[0],
        'f_ci95_upper': f_ci95[1],
        'f_range_used': str(f_range)
    })

df_results = pd.DataFrame(results)

# 显示结果
print("=== 反向反演 + 观测不确定性传播结果 ===\n")
cols_show = ['sample_id', 'age_ma', 'delta15N', 'delta15N_std',
             'f_central', 'f_mean', 'f_std']
print(df_results[cols_show].to_string(index=False, float_format='%.3f'))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ages = df_results['age_ma'].values
f_mean = df_results['f_mean'].values
f_std = df_results['f_std'].values
f_lower68 = df_results['f_ci68_lower'].values
f_upper68 = df_results['f_ci68_upper'].values

# 绘制误差棒
ax.errorbar(ages, f_mean, yerr=[f_mean - f_lower68, f_upper68 - f_mean],
            fmt='o', color='steelblue', ecolor='steelblue',
            capsize=4, capthick=1.5, markersize=7,
            linewidth=1.5, label='Observed (68% CI)', zorder=5)

# 填充 800 Ma 分界线
ax.axvline(800, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='800 Ma boundary')
ax.axvspan(800, 860, alpha=0.08, color='red', label='Pre-800 Ma (anoxic)')
ax.axvspan(740, 800, alpha=0.08, color='green', label='Post-800 Ma (oxic)')

# 添加趋势线（简单滑动平均）
sort_idx = np.argsort(ages)
ax.plot(ages[sort_idx], f_mean[sort_idx], '--', color='gray',
        linewidth=1, alpha=0.6, label='Trend')

# 标注样品编号
for _, row in df_results.iterrows():
    ax.annotate(row['sample_id'],
                xy=(row['age_ma'], row['f_mean']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=7, alpha=0.7)

ax.set_xlabel('Age (Ma)', fontsize=12)
ax.set_ylabel('f$_{assimilator}$', fontsize=12)
ax.set_title('Neoproterozoic Nitrate Availability from δ¹⁵N Data', fontsize=12)
ax.set_xlim(795, 855)
ax.set_ylim(-0.05, 0.55)
ax.invert_xaxis()  # 年龄从左到右递减（地质惯例）
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('nitrogen_data_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ 图片已保存: nitrogen_data_timeseries.png")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# 绘制理论关系曲线（重新计算，确保存在）
curve_full = n_neoproterozoic.calculate_f_assimilator_curve(
    f_range=(0.0, 1.0), n_points=100, n_monte_carlo=2000)

ax.plot(curve_full['f_assimilator'], curve_full['delta15N_sed_mean'],
        'b-', linewidth=1.5, alpha=0.6, label='Model curve')
ax.fill_between(curve_full['f_assimilator'],
                curve_full['delta15N_sed_ci68_lower'],
                curve_full['delta15N_sed_ci68_upper'],
                alpha=0.2, color='blue')

# 按年龄着色标注实测点
# 年龄大的偏红（早期，缺氧），年龄小的偏绿（晚期，氧化）
ages_norm = (df_results['age_ma'] - df_results['age_ma'].min()) / \
            (df_results['age_ma'].max() - df_results['age_ma'].min())
colors = plt.cm.RdYlGn(1 - ages_norm)  # 红->绿

for _, row in df_results.iterrows():
    ax.errorbar(row['f_mean'], row['delta15N'],
                xerr=[[row['f_mean'] - row['f_ci68_lower']],
                      [row['f_ci68_upper'] - row['f_mean']]],
                fmt='o', color=colors[_], ecolor='gray',
                capsize=3, markersize=8,
                markeredgecolor='black', markeredgewidth=0.5,
                zorder=5)

# 添加颜色条
sm = plt.cm.ScalarMappable(cmap='RdYlGn_r',
                           norm=plt.Normalize(vmin=df_results['age_ma'].min(),
                                              vmax=df_results['age_ma'].max()))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label('Age (Ma)', rotation=270, labelpad=15)

ax.set_xlabel('f$_{assimilator}$', fontsize=12)
ax.set_ylabel('δ¹⁵N$_{sed}$ (‰)', fontsize=12)
ax.set_title('Observed Data on Model Curve', fontsize=12)
ax.set_xlim(-0.05, 0.55)
ax.set_ylim(-1, 7)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('nitrogen_data_on_curve.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ 图片已保存: nitrogen_data_on_curve.png")


In [ ]:
# 导出详细结果
df_results.to_csv('nitrogen_data_inversion_results.csv', index=False)
print("✓ 结果已保存: nitrogen_data_inversion_results.csv")

# 同时保存为 Excel（包含原始数据和反演结果）
with pd.ExcelWriter('nitrogen_data_inversion_results.xlsx') as writer:
    df_results.to_excel(writer, sheet_name='Inversion_Results', index=False)
    df_data.to_excel(writer, sheet_name='Original_Data', index=False)
print("✓ 结果已保存: nitrogen_data_inversion_results.xlsx")

# 打印最终汇总
print("\n=== 分析汇总 ===")
print(f"平均 f_assimilator: {df_results['f_mean'].mean():.3f} ± {df_results['f_mean'].std():.3f}")
print(f"早期 (>830 Ma) 平均 f: {df_results[df_results['age_ma'] > 830]['f_mean'].mean():.3f}")
print(f"晚期 (<830 Ma) 平均 f: {df_results[df_results['age_ma'] <= 830]['f_mean'].mean():.3f}")


---

## 11. 只有深度数据（无年龄）时的处理

如果实测数据只有**深度/层位**而没有绝对年龄，分析逻辑几乎相同，只需做以下调整：

1. **X 轴改为深度**：把 `age_ma` 替换为 `depth_m`（或 `stratigraphic_height_m`）
2. **不再按年龄分段选 f_range**：深度本身不直接约束氧化状态，需根据地质背景统一选择合理范围，或按深度段手动分段
3. **深度方向**：地质惯例通常向下为正（深度增加），或向上为正（地层高度）。绘图时根据实际情况决定是否反转 X 轴

下面的示例用现有数据演示（将 `age_ma` 列视为 `depth_m`）。如果你有自己的深度数据文件，只需修改读取路径和列名即可。


In [ ]:
# ============================================================
# 读取数据并适配为深度格式
# ============================================================

# 如果有独立的深度数据文件，直接修改这里：
# df_depth = pd.read_excel('your_depth_data.xlsx')

# 这里用现有数据做演示，把 age_ma 视为 depth_m
df_depth = df_data.copy()
df_depth = df_depth.rename(columns={'age_ma': 'depth_m'})

# 如果你有独立的深度文件，确保列名为：
#   sample_id, depth_m, delta15N, delta15N_std, lithology (可选)

print(df_depth.to_string(index=False))
print(f"\n共 {len(df_depth)} 个样品")
print(f"深度范围: {df_depth['depth_m'].min():.0f} – {df_depth['depth_m'].max():.0f} m")


In [ ]:
# ============================================================
# 深度数据反向反演（统一 f_range）
# ============================================================

# 深度本身不提供氧化/还原约束，因此统一使用较宽的 f_range
# 如果你的地质背景已知（如下部缺氧、上部氧化），可手动分段：
#   deep_section  -> f_range = (0.0, 0.30)
#   shallow_section -> f_range = (0.0, 0.60)

f_range_uniform = (0.0, 0.50)  # 统一范围
n_mc_obs = 2000

results_depth = []
for _, row in df_depth.iterrows():
    delta = row['delta15N']
    std = row['delta15N_std']
    depth = row['depth_m']
    sid = row['sample_id']
    
    # 示例：按深度手动分段（取消注释即可使用）
    # if depth > 830:
    #     f_range = (0.0, 0.30)   # 深部：推测缺氧
    # else:
    #     f_range = (0.0, 0.60)   # 浅部：推测氧化
    f_range = f_range_uniform
    
    # 中心值反演
    central = n_neoproterozoic.inverse_model(delta, f_range=f_range)
    
    # 蒙特卡洛传播观测不确定性
    delta_samples = np.random.normal(delta, std, n_mc_obs)
    f_samples = []
    for d_samp in delta_samples:
        if d_samp < -2 or d_samp > 10:
            continue
        try:
            res = n_neoproterozoic.inverse_model(d_samp, f_range=f_range)
            f_samples.append(res['f_assimilator'])
        except Exception:
            continue
    
    f_samples = np.array(f_samples)
    f_mean = float(np.mean(f_samples))
    f_std = float(np.std(f_samples))
    f_ci68 = (float(np.percentile(f_samples, 16)),
              float(np.percentile(f_samples, 84)))
    f_ci95 = (float(np.percentile(f_samples, 2.5)),
              float(np.percentile(f_samples, 97.5)))
    
    results_depth.append({
        'sample_id': sid,
        'depth_m': depth,
        'delta15N': delta,
        'delta15N_std': std,
        'lithology': row.get('lithology', ''),
        'f_central': float(central['f_assimilator']),
        'f_mean': f_mean,
        'f_std': f_std,
        'f_ci68_lower': f_ci68[0],
        'f_ci68_upper': f_ci68[1],
        'f_ci95_lower': f_ci95[0],
        'f_ci95_upper': f_ci95[1],
        'f_range_used': str(f_range)
    })

df_results_depth = pd.DataFrame(results_depth)

print("=== 深度数据反向反演结果（统一 f_range） ===\n")
cols_show = ['sample_id', 'depth_m', 'delta15N', 'delta15N_std',
             'f_central', 'f_mean', 'f_std']
print(df_results_depth[cols_show].to_string(index=False, float_format='%.3f'))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

depths = df_results_depth['depth_m'].values
f_mean = df_results_depth['f_mean'].values
f_lower68 = df_results_depth['f_ci68_lower'].values
f_upper68 = df_results_depth['f_ci68_upper'].values

ax.errorbar(depths, f_mean,
            yerr=[f_mean - f_lower68, f_upper68 - f_mean],
            fmt='o', color='steelblue', ecolor='steelblue',
            capsize=4, capthick=1.5, markersize=7,
            linewidth=1.5, label='Observed (68% CI)', zorder=5)

# 趋势线
sort_idx = np.argsort(depths)
ax.plot(depths[sort_idx], f_mean[sort_idx], '--', color='gray',
        linewidth=1, alpha=0.6, label='Trend')

# 样品标注
for _, row in df_results_depth.iterrows():
    ax.annotate(row['sample_id'],
                xy=(row['depth_m'], row['f_mean']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=7, alpha=0.7)

# 深度轴方向：地质惯例通常是深度向下增加，无需反转
# 如果你的数据是地层高度（向上增加），取消下面注释反转 X 轴：
# ax.invert_xaxis()

ax.set_xlabel('Depth (m)', fontsize=12)
ax.set_ylabel('f$_{assimilator}$', fontsize=12)
ax.set_title('Nitrate Availability from Depth Profile', fontsize=12)
ax.set_ylim(-0.05, 0.55)
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('nitrogen_depth_profile.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ 图片已保存: nitrogen_depth_profile.png")


In [ ]:
# 双轴图：深度 vs δ¹⁵N（左轴）和 f（右轴），便于对比
fig, ax1 = plt.subplots(figsize=(9, 5))

depths = df_results_depth['depth_m'].values
sort_idx = np.argsort(depths)

# 左轴：δ¹⁵N
color1 = 'tab:red'
ax1.set_xlabel('Depth (m)', fontsize=12)
ax1.set_ylabel('δ¹⁵N$_{sed}$ (‰)', color=color1, fontsize=12)
ax1.errorbar(depths, df_results_depth['delta15N'],
             yerr=df_results_depth['delta15N_std'],
             fmt='s', color=color1, ecolor=color1,
             capsize=4, markersize=7, label='δ¹⁵N')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_ylim(-1, 7)

# 右轴：f_assimilator
ax2 = ax1.twinx()
color2 = 'tab:blue'
ax2.set_ylabel('f$_{assimilator}$', color=color2, fontsize=12)
ax2.errorbar(depths, df_results_depth['f_mean'],
             yerr=[df_results_depth['f_mean'] - df_results_depth['f_ci68_lower'],
                   df_results_depth['f_ci68_upper'] - df_results_depth['f_mean']],
             fmt='o', color=color2, ecolor=color2,
             capsize=4, markersize=7, label='f (68% CI)')
ax2.tick_params(axis='y', labelcolor=color2)
ax2.set_ylim(-0.05, 0.55)

# 图例
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

ax1.set_title('Depth Profile: δ¹⁵N and Inferred Nitrate Availability', fontsize=12)
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('nitrogen_depth_dual_axis.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ 图片已保存: nitrogen_depth_dual_axis.png")


In [ ]:
# 导出深度数据反演结果
df_results_depth.to_csv('nitrogen_depth_inversion_results.csv', index=False)
print("✓ 结果已保存: nitrogen_depth_inversion_results.csv")

with pd.ExcelWriter('nitrogen_depth_inversion_results.xlsx') as writer:
    df_results_depth.to_excel(writer, sheet_name='Depth_Inversion', index=False)
    df_depth.to_excel(writer, sheet_name='Original_Data', index=False)
print("✓ 结果已保存: nitrogen_depth_inversion_results.xlsx")

print("\n=== 深度数据分析汇总 ===")
print(f"平均 f_assimilator: {df_results_depth['f_mean'].mean():.3f} ± {df_results_depth['f_mean'].std():.3f}")
print(f"最大 f: {df_results_depth['f_mean'].max():.3f} (深度 {df_results_depth.loc[df_results_depth['f_mean'].idxmax(), 'depth_m']:.0f} m)")
print(f"最小 f: {df_results_depth['f_mean'].min():.3f} (深度 {df_results_depth.loc[df_results_depth['f_mean'].idxmin(), 'depth_m']:.0f} m)")
